# Supervised Fine-Tuning (SFT) with Serverless Customization on SageMaker AI

## Lab 1 – Prepare the Dataset

This is the first of four interconnected labs that take you from raw data to a deployed, fine-tuned large language model:

| Lab | Notebook | What you'll do |
|-----|----------|----------------|
| **Lab 1** | `1-prepare-data.ipynb` ← *you are here* | Stream a multilingual reasoning dataset, reformat it into the SFT schema, and register it in SageMaker AI |
| **Lab 2** | `2-fine-tune-llm.ipynb` | Submit a serverless LoRA fine-tuning job on SageMaker AI and register the result in the Model Registry |
| **Lab 3** | `3-evaluation.ipynb` | Evaluate the fine-tuned model using LLM-as-Judge with custom metrics |
| **Lab 4** | `4-deployment.ipynb` | Deploy the merged model to a real-time SageMaker endpoint powered by vLLM |

### What you'll do in this lab

In supervised fine-tuning, the model learns *purely from the examples you give it* — so data quality and format directly determine fine-tuning quality. In this lab you will:

1. **Stream** the [Multilingual-Thinking](https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking) dataset from Hugging Face Hub — a collection of multilingual reasoning problems with explicit chain-of-thought traces
2. **Split** the data into train, validation, and test sets
3. **Reformat** each example into the `prompt` / `completion` schema that SageMaker SFT expects, preserving reasoning traces inside `<think>…</think>` tags
4. **Upload** the prepared splits to Amazon S3
5. **Register** them as versioned `DataSet` assets in the SageMaker AI Registry for lineage tracking

***

### Prerequisites

Before running this notebook, make sure you have:
- An AWS account with SageMaker access
- A SageMaker Studio domain or SageMaker Notebook Instance (this notebook was tested on `ml.t3.medium`)
- An IAM execution role with `AmazonSageMakerFullAccess` and S3 read/write permissions

***

### Step 1 – Install requirements

Run the cell below to install all Python dependencies, including:
- **`datasets`** — Hugging Face library for streaming and processing datasets
- **`pandas` / `scikit-learn`** — data manipulation and train/val/test splitting
- **`sagemaker`** — AWS SageMaker Python SDK for session management, training, and registry operations

In [ ]:
%pip install -r requirements.txt

***

### Step 2 – Set up the SageMaker session

We start by creating a **SageMaker `Session`** — a lightweight helper that manages the connection to your AWS account. It:
- Resolves the **default S3 bucket** for staging datasets and model artifacts
- Reads your **IAM execution role** — the identity that grants SageMaker permission to read S3, write metrics, and launch training jobs on your behalf

> **Tip:** If you're running inside SageMaker Studio, `get_execution_role()` automatically retrieves the Studio execution role. Outside Studio, you can create a role named `sagemaker_execution_role` in IAM with the `AmazonSageMakerFullAccess` managed policy attached.

#### Setup and dependencies

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

***

### Step 3 – Prepare the dataset

#### About the dataset

We use the [**Multilingual-Thinking**](https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking) dataset from Hugging Face Hub. This dataset was specifically designed to train models to externalize their reasoning across multiple languages. Each example contains:

| Field | Description |
|-------|-------------|
| `messages` | A list of chat turns: `system`, `user`, and `assistant` |
| `reasoning_language` | The non-English language the model should *think* in (e.g., French, Spanish, Chinese) |
| `thinking` | The model's explicit chain-of-thought reasoning in the target language |

#### The fine-tuning goal

The goal of this workshop is to fine-tune **NVIDIA Nemotron 3 Nano 30B** — a highly capable, mixture-of-experts (MoE) model to:
1. **Reason** inside `<think>…</think>` tags in a target non-English language (specified via the system prompt)
2. **Answer** the question in fluent English

This is a form of *reasoning-language alignment*: the model already has strong reasoning abilities from pre-training, but we're steering it to externalize its chain-of-thought in a consistent, structured format — making its reasoning process transparent and parseable by downstream applications.

To learn more about [NVIDIA Nemotron 3 Nano 30B](https://build.nvidia.com/nvidia/nemotron-3-nano-30b) — a 30B active parameter MoE model designed for efficient, high-quality inference.

#### Load the full dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "HuggingFaceH4/Multilingual-Thinking",
    split="train",
)

dataset

In [ ]:
import pandas as pd

df = pd.DataFrame(dataset)

df.head()

#### Split into train / validation / test

We divide the dataset into three disjoint sets:

| Split | Proportion | Purpose |
|-------|-----------|---------|
| **Train** | 70% | The examples the model learns from during fine-tuning |
| **Validation** | 20% | Held-out set used *during* training to detect overfitting early |
| **Test** | 10% | Reserved for final evaluation in Lab 3 — never seen during training |


In [ ]:
from sklearn.model_selection import train_test_split

train, val = train_test_split(df, test_size=0.2, random_state=42)
train, test = train_test_split(train, test_size=0.125, random_state=42)

print("Number of train elements: ", len(train))
print("Number of validation elements: ", len(val))
print("Number of test elements: ", len(test))

#### Reformat into the SFT schema

SageMaker Serverless Fine-Tuning expects each **training and validation** example to contain exactly two fields:

- **`prompt`** — the model's input (the user's question, framed with the system instruction)
- **`completion`** — the target output the model should learn to produce

For our task, the `completion` field wraps the chain-of-thought reasoning in `<think>…</think>` tags, followed by the English final answer:

```
<think>
[chain-of-thought reasoning in the target non-English language]
</think>

[final answer in English]
```

**Why `<think>` tags?** These delimiters teach the model a *separable reasoning-then-answer* structure — keeping the chain-of-thought and the final answer distinct so they can be processed and displayed independently.

We will also create a test split that uses `query` and `response` — the format needed for evaluation later in this workshop.

In [ ]:
from datasets import Dataset
import textwrap
from tqdm import tqdm


def prepare_dataset_train_val(sample):
    system = None
    prompt = None
    completion = None

    for el in sample["messages"]:
        if el["role"] == "system":
            system_prompt = """
            You are an AI assistant that thinks in {language} but responds in English.

            IMPORTANT: Follow this exact format for every response:
            1. First, write your reasoning and thoughts inside <think>...</think> tags
            2. Then, provide your final answer in English

            Always think through the problem in {language}, then translate your conclusion to English for the final response.
            """

            system_prompt = system_prompt.format(language=sample["reasoning_language"])
            system_prompt = textwrap.dedent(system_prompt).strip()
            system = system_prompt
        elif el["role"] == "user":
            prompt = el["content"]
        else:
            thinking = el.get("thinking")
            if thinking is not None and thinking != "" and thinking != "null":
                completion = f"<think>\n{thinking}\n</think>\n\n"

            completion += el["content"]
    yield {
        "system": system, 
        "prompt": prompt,
        "completion": completion,
    }


def prepare_dataset_test(sample):
    # The evaluation job (3-evaluation.ipynb) consumes the GenQA format, whose
    # columns are `query` (required), `response` (the reference answer, exposed to
    # the judge as {{ground_truth}}) and `system` (optional, passed to the model at
    # inference time). This differs from the SFT prompt/completion format used for
    # train/val above, so the test split emits GenQA keys instead.
    system = None
    query = None
    response = None

    for el in sample["messages"]:
        if el["role"] == "system":
            system_prompt = """
            You are an AI assistant that thinks in {language} but responds in English.

            IMPORTANT: Follow this exact format for every response:
            1. First, write your reasoning and thoughts inside <think>...</think> tags
            2. Then, provide your final answer in English

            Always think through the problem in {language}, then translate your conclusion to English for the final response.
            """

            system_prompt = system_prompt.format(language=sample["reasoning_language"])
            system_prompt = textwrap.dedent(system_prompt).strip()
            system = system_prompt
        elif el["role"] == "user":
            query = el["content"]
        else:
            thinking = el.get("thinking")
            if thinking is not None and thinking != "" and thinking != "null":
                response = f"<think>\n{thinking}\n</think>\n\n"

            response += el["content"]
    yield {
        "system": system,
        "query": query,
        "response": response,
    }

In [ ]:
def convert_to_messages_train_val(dataset):
    """Iteratively run conversion on multi-turn conversation and flatten to messages"""
    records = []

    print("Original length: ", len(dataset))

    # Unroll your generator for every dataset row
    for row in tqdm(dataset, total=len(dataset), desc="Converting to messages"):
        for example in prepare_dataset_train_val(row):
            records.append(example)

    # Convert list of dicts → Hugging Face Dataset and return
    return Dataset.from_list(records)


def convert_to_messages_test(dataset):
    """Iteratively run conversion on multi-turn conversation and flatten to messages"""
    records = []

    print("Original length: ", len(dataset))

    # Unroll your generator for every dataset row
    for row in tqdm(dataset, total=len(dataset), desc="Converting to messages"):
        for example in prepare_dataset_test(row):
            records.append(example)

    # Convert list of dicts → Hugging Face Dataset and return
    return Dataset.from_list(records)

In [ ]:
from datasets import Dataset, DatasetDict
import json
from random import randint

train_dataset = Dataset.from_pandas(train)
val_dataset = Dataset.from_pandas(val)
test_dataset = Dataset.from_pandas(test)

dataset = DatasetDict(
    {"train": train_dataset, "val": val_dataset, "test": test_dataset}
)

train_dataset = convert_to_messages_train_val(dataset["train"])

print(json.dumps(train_dataset[randint(0, len(train_dataset) - 1)], indent=2))

val_dataset = convert_to_messages_train_val(dataset["val"])

test_dataset = convert_to_messages_test(dataset["test"])

print(json.dumps(test_dataset[randint(0, len(test_dataset) - 1)], indent=2))

### Step 4 – Upload the splits to Amazon S3

SageMaker Serverless Fine-Tuning reads all training inputs from **Amazon S3**. We write each split to [JSON Lines](https://jsonlines.org/) (`.jsonl`) format — one JSON object per line — and upload it to the session's default bucket.


Local copies are deleted after upload to keep the notebook environment clean.

In [ ]:
import shutil

In [ ]:
if default_prefix:
    input_path = f"{default_prefix}/datasets/serverless-model-customization-sft"
else:
    input_path = f"datasets/serverless-model-customization-sft"

train_dataset_s3_path = f"s3://{bucket_name}/{input_path}/train/dataset.jsonl"
val_dataset_s3_path = f"s3://{bucket_name}/{input_path}/val/dataset.jsonl"
test_dataset_s3_path = f"s3://{bucket_name}/{input_path}/test/dataset.jsonl"

In [ ]:
train_dataset.to_json("./data/train/dataset.jsonl", orient="records")
val_dataset.to_json("./data/val/dataset.jsonl", orient="records")
test_dataset.to_json("./data/test/dataset.jsonl", orient="records")

s3_client.upload_file(
    "./data/train/dataset.jsonl", bucket_name, f"{input_path}/train/dataset.jsonl"
)
s3_client.upload_file(
    "./data/val/dataset.jsonl", bucket_name, f"{input_path}/val/dataset.jsonl"
)
s3_client.upload_file(
    "./data/test/dataset.jsonl", bucket_name, f"{input_path}/test/dataset.jsonl"
)

shutil.rmtree("./data")

print(f"Training data uploaded to:")
print(train_dataset_s3_path)
print(val_dataset_s3_path)
print(test_dataset_s3_path)

### Step 5 – Register the datasets in the SageMaker AI Registry

Registering the datasets as **`DataSet` assets** in the SageMaker AI Registry provides two key benefits:

1. **Versioning** — Each registration creates an immutable snapshot. If you update the dataset later, you can track exactly which version was used for each training run.
2. **Lineage tracking** — SageMaker automatically records the dataset-to-model relationship. In the console, you can trace any registered model all the way back to the exact dataset it was trained on.

The `customization_technique=CustomizationTechnique.SFT` tag marks the train and validation sets as SFT inputs — this is required for the `SFTTrainer` in Lab 2 to accept them. The test set is registered without this tag since it serves evaluation purposes only.

In [ ]:
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.ai_registry.dataset_utils import CustomizationTechnique

In [ ]:
dataset_train = DataSet.create(
    name="Multilingual-Thinking-sft-train",
    source=train_dataset_s3_path,
    customization_technique=CustomizationTechnique.SFT,
    wait=True,
)

print(f"TRAINING_DATASET ARN: {dataset_train.arn}")

dataset_val = DataSet.create(
    name="Multilingual-Thinking-sft-val",
    source=val_dataset_s3_path,
    customization_technique=CustomizationTechnique.SFT,
    wait=True,
)

print(f"VALIDATION_DATASET ARN: {dataset_val.arn}")

dataset_test = DataSet.create(
    name="Multilingual-Thinking-sft-test",
    source=test_dataset_s3_path,
    wait=True,
)

print(f"TEST_DATASET ARN: {dataset_test.arn}")